# Eval set v3 — natural phrasing, same ground truth

## Why

Phase 13 (`scripts/19. embeddings_evaluation_v2.ipynb`) compared seven embedding models and found the ranking was almost meaningless: **five of the seven scored recall@5 = 1.000**, and the total disagreement across seven architecturally different models was **four questions out of sixty**. `eval_v2` had run out of power to tell good retrievers apart.

The diagnosis was in how v2 was written — **by reading the articles**. That left the questions echoing the statute's own vocabulary, so each question sits close to a paraphrase of its own answer:

| v2 (echoes the statute) | the article it points to |
|---|---|
| "What are the conditions for entitlement to the unemployment allowance?" | SSL Art 50, headed *conditions for entitlement to the unemployment allowance* |
| "كم يوماً من الإجازة المرضية يستحق العامل في السنة الواحدة؟" | Labor Art 65, which reads *يستحق العامل ... إجازة مرضية* |

Supporting evidence from Phase 13's truncation control: `multilingual-e5-large-instruct` had **50 of 531 articles cut off** at its hard 512-token limit and still placed third, ahead of two models that read every article in full. If discarding the tail of ~9% of the corpus costs nothing measurable, the matching is happening on opening lines — exactly where a question written *from* an article would point.

## What changes, and what does not

**Only the wording of `question_text`.** Every `answer_law`, `answer_article_numbers`, `answer_language`, `question_category`, `difficulty` and `labeling_note` is carried across unchanged and asserted identical. This is a rephrasing, not a new eval set — so v3 scores are directly comparable to v2 scores, and the difference between them is a clean measurement of how much the statute-echo was inflating the numbers.

The target is **natural, not colloquial**. People type more formally than they speak, and the original decision to use formal register was right. What changes is that questions now describe a *situation* in ordinary words instead of naming the legal concept:

- ❌ "What are the conditions for entitlement to the unemployment allowance?"
- ✅ "I lost my job. Do I qualify for unemployment payments?"
- ❌ "كم يوماً من الإجازة المرضية يستحق العامل في السنة الواحدة؟"
- ✅ "كم يوماً أستطيع أن أتغيب بسبب المرض في السنة؟"

Both are grammatical formal Arabic/English. Neither is slang.

## Versioning

`eval_v2.jsonl` is **left intact**, exactly as `eval_draft.jsonl` (v1) was kept when v2 was built. Phase 13's published numbers were measured on v2, and overwriting it would silently make a documented result unreproducible. v3 is a new file.

Each row also keeps the v2 wording in a `question_text_statute` field. The corpus vectors are already cached from Phase 13, so re-scoring every model on v3 costs only 60 new query embeddings per model — fractions of a cent — and the paired comparison comes for free.

In [1]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
V2_PATH = PROJECT / "eval" / "eval_v2.jsonl"
CHUNKS_PATH = PROJECT / "0. cleaned_data" / "chunks_articles.parquet"

v2 = [json.loads(l) for l in open(V2_PATH, encoding="utf-8")]
chunks = pd.read_parquet(CHUNKS_PATH)
chunks["article_number"] = chunks["article_number"].astype(str)
print(f"loaded {len(v2)} v2 questions, {len(chunks)} articles")

loaded 60 v2 questions, 531 articles


## The rewrites

One entry per question. The rewrite describes the asker's situation in ordinary words; the ground truth it points to is untouched.

In [2]:
REWRITES = {
    # ---------- Social Security Law, English ----------
    "q01": "I was injured while at work. What does social security actually cover for me?",
    "q02": "I need an artificial limb and I have to travel for treatment. Will those costs be paid for me?",
    "q03": "A worker at my company was hurt on site today. What am I required to do straight away?",
    "q04": "I cannot work while I recover from an injury I got at work. Will I still be paid, and how much?",
    "q05": "If someone is killed or left permanently unable to work because of an accident at work, what money is paid and how is the amount worked out?",
    "q06": "Can someone lose their right to injury money because of the way they behaved?",
    "q07": "Can I also sue my employer for more than what social security pays me for my injury?",
    "q08": "I got ill from a job I left a while ago, and the symptoms only showed up now. Am I still covered?",
    "q09": "Can I take a paid job while I am receiving a disability pension for a work injury?",
    "q10": "I am pregnant and working. Do I qualify for a maternity payment, and how much would it be?",
    "q11": "I just lost my job. Do I qualify for unemployment payments?",
    "q12": "How long do unemployment payments keep coming, and does the amount drop over time?",
    "q13": "Can I draw unemployment payments at the same time as a maternity payment or a retirement pension?",
    "q14": "At what age can I stop working and draw a pension, and how many contributions do I need?",
    "q15": "Can I stop working before the normal retirement age, and how would my pension be worked out?",

    # ---------- Social Security Law, Arabic ----------
    "q16": "هل يُخصم من راتبي شيء مقابل التأمين ضد إصابات العمل، أم يدفعه صاحب العمل وحده؟",
    "q17": "أُصبت في العمل وما زلت أتعالج. إلى متى تستمر المؤسسة في تغطية علاجي؟",
    "q18": "أُصبت مرة أخرى في العمل بعد إصابة قديمة. كيف يُحسب ما أستحقه هذه المرة؟",
    "q19": "بعد أن تقرر لي راتب بسبب العجز، هل يمكن أن يُعاد فحصي طبياً مرة أخرى، ومتى؟",
    "q20": "لم أقتنع بقرار اللجنة الطبية في إصابتي. هل أستطيع الاعتراض، وكم من الوقت أمامي؟",
    "q21": "تسبب لي شخص آخر بإصابة أثناء عملي. هل تدفع لي المؤسسة أم أطالب ذلك الشخص؟",
    "q22": "أتقاضى بدل إجازة الأمومة وعُرض علي عمل خلال الإجازة. هل يتوقف البدل إن قبلت العمل؟",
    "q23": "وجدت عملاً جديداً وأنا ما زلت أتقاضى بدل التعطل. ماذا يحدث للبدل؟",
    "q24": "متى أستلم أول دفعة من بدل التعطل بعد أن تركت عملي؟",
    "q25": "هل أستطيع طلب بدل التعطل أكثر من مرة خلال حياتي العملية؟",
    "q26": "خرجت نهائياً من الضمان. ماذا يحدث للمبالغ التي اقتُطعت مني لتأمين التعطل؟",
    "q27": "بلغت الستين ولم تكتمل اشتراكاتي اللازمة للراتب التقاعدي. هل أستطيع الاستمرار في الضمان؟",
    "q28": "تنقصني مدة قصيرة حتى أستحق الراتب التقاعدي. هل أستطيع شراء هذه المدة بمبلغ أدفعه؟",
    "q29": "توفي زوجي وهو على رأس عمله. ما الشروط حتى يستحق ورثته راتباً؟",
    "q30": "أصابني عجز بسبب مرض لا علاقة له بعملي. هل أستحق راتباً من الضمان؟",

    # ---------- Labour Law 1996, English ----------
    "q31": "What time off can I take during the year, and which of it is paid?",
    "q32": "How much warning must be given before ending a job that has no fixed end date?",
    "q33": "I think I was fired for no good reason. What can I get if that is proven?",
    "q34": "My contract had a fixed end date but my employer ended it early. What am I owed?",
    "q35": "Are there situations where an employer simply is not allowed to end someone's job?",
    "q36": "When can an employer dismiss someone on the spot, with no warning at all?",
    "q37": "I am not covered by social security. What is my employer supposed to pay me when my job ends?",
    "q38": "How long can an employer keep me on trial at the start of a job, and what happens during that time?",
    "q39": "When must my wages reach me, and what is my employer allowed to take out of them?",
    "q40": "How many hours can I be asked to work, and what extra pay do I get for long hours or for working on a holiday?",
    "q41": "I did not use all my annual leave before leaving the job. Do I get paid for the days I did not take?",
    "q42": "How long can I stay off work after giving birth, and do I get time during the day for breastfeeding afterwards?",
    "q43": "If a worker is killed or left permanently unable to work by an accident at work, what is the employer required to pay?",

    # ---------- Labour Law 1996, Arabic ----------
    "q44": "كم يوم إجازة سنوية لي، وهل أستطيع تقسيمها، وماذا يحدث إن لم أستعملها؟",
    "q45": "كم يوماً أستطيع أن أتغيب بسبب المرض في السنة الواحدة؟",
    "q46": "هل هناك إجازات أخرى أستطيع أخذها غير السنوية والمرضية؟",
    "q47": "أريد التفرغ لتربية أطفالي. هل أستطيع أخذ إجازة لذلك، وبأي شروط؟",
    "q48": "نُقل زوجي للعمل في محافظة أخرى. هل أستطيع أخذ إجازة حتى أرافقه؟",
    "q49": "هل أستطيع ترك عملي فوراً دون إشعار وأحتفظ بحقوقي، وفي أي حالات؟",
    "q50": "انتهى عملي. هل يلزم صاحب العمل أن يعطيني ورقة تثبت أنني عملت لديه؟",
    "q51": "شركتنا تمر بضائقة وتريد تقليص عدد الموظفين. ما الذي يلزمها قبل أن تفعل ذلك؟",
    "q52": "إذا توفي العامل، لمن تُدفع المبالغ المستحقة له؟",
    "q53": "أتلفت آلة في العمل من غير قصد. هل يُخصم ذلك من راتبي، وكم أقصى ما يمكن خصمه؟",
    "q54": "أوقف صاحب العمل العمل مؤقتاً لسبب خارج عن إرادته. هل أتقاضى أجري في تلك الفترة؟",
    "q55": "كم يوم راحة لي في الأسبوع، وهل يكون بأجر؟",
    "q56": "ما الذي يلزم صاحب العمل حتى يحميني من مخاطر العمل، وما الذي يلزمني أنا؟",

    # ---------- Professional Work Law 2019 ----------
    "q57": "Can I work in a place that has not been given a classification yet?",
    "q58": "Am I put into a category as a professional worker, and what does that category affect?",
    "q59": "هل أستطيع ممارسة مهنتي في محل لم يُصنَّف بعد؟",
    "q60": "هل يُوضع العاملون المهنيون في فئات، وما الذي يترتب على ذلك؟",
}

assert len(REWRITES) == 60, len(REWRITES)
assert set(REWRITES) == {r["question_id"] for r in v2}, "question id mismatch"
print(f"{len(REWRITES)} rewrites defined")

60 rewrites defined


## Assemble

Ground-truth fields are copied across, then asserted identical field-by-field. A rewrite that quietly changed which article a question points to would be far worse than no rewrite at all.

In [3]:
v3 = []
for r in v2:
    new = dict(r)
    new["question_text"] = REWRITES[r["question_id"]]
    new["question_text_statute"] = r["question_text"]      # v2 wording, kept for the paired comparison
    v3.append(new)

GROUND_TRUTH_FIELDS = ["question_id", "question_language", "answer_law", "answer_article_numbers",
                       "answer_language", "question_category", "difficulty", "n_articles",
                       "labeling_note", "source_article_excerpt"]

for old, new in zip(v2, v3):
    for f in GROUND_TRUTH_FIELDS:
        assert old[f] == new[f], f"{old['question_id']}: field '{f}' changed"

# every rewrite must actually be a rewrite
unchanged = [r["question_id"] for r in v3 if r["question_text"] == r["question_text_statute"]]
assert not unchanged, f"not rewritten: {unchanged}"

# and must still point at articles that exist in the corpus
have = {(c.law, c.language, c.article_number) for c in chunks.itertuples()}
missing = [(r["question_id"], a) for r in v3 for a in r["answer_article_numbers"]
           if (r["answer_law"], r["answer_language"], str(a)) not in have]
assert not missing, f"ground truth absent from corpus: {missing}"

print(f"{len(v3)} questions rewritten, all ground truth verified unchanged")

60 questions rewritten, all ground truth verified unchanged


## Did the statute-echo actually go down?

The whole point of v3 is that questions should stop reusing the wording of the article they point to. That is measurable rather than a matter of taste: tokenise each question and its gold article, and compute what fraction of the question's content words also appear in the article.

Arabic is normalised the same way `src/retrieval.py` does it (diacritics stripped, alef/yaa/ta-marbuta folded), so the overlap figure is not distorted by orthographic variants. Stopwords are removed on both sides — matching on "the" or "من" would tell us nothing.

If v3's overlap is not clearly lower than v2's, the rewrite failed and should be redone rather than shipped.

In [4]:
AR_DIACRITICS = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670\u0640]")
WORD_RE = re.compile(r"\w+", re.UNICODE)

EN_STOP = {"what", "is", "are", "the", "a", "an", "of", "to", "for", "in", "on", "at", "and", "or",
           "do", "does", "did", "can", "may", "must", "i", "my", "me", "it", "that", "this", "be",
           "am", "was", "were", "been", "have", "has", "had", "will", "would", "if", "not", "no",
           "you", "your", "they", "their", "he", "she", "his", "her", "we", "our", "so", "than",
           "how", "when", "where", "which", "who", "whom", "there", "any", "all", "some", "get",
           "got", "with", "from", "by", "as", "but", "out", "up", "about", "into", "over", "after",
           "before", "during", "while", "still", "just", "only", "also", "much", "many", "more"}
AR_STOP = {"من", "في", "على", "الى", "عن", "مع", "هل", "ما", "ماذا", "متى", "اين", "كيف", "كم",
           "الذي", "التي", "هذا", "هذه", "ذلك", "تلك", "و", "او", "ان", "انه", "اذا", "لا", "لم",
           "لن", "قد", "كان", "يكون", "هو", "هي", "انا", "لي", "له", "لها", "بعد", "قبل", "عند",
           "حتى", "ثم", "كل", "بعض", "غير", "بين", "دون", "خلال", "اثناء", "لدي", "الا", "ايضا",
           "الذين", "وهل", "وما", "به", "بها", "منه", "منها", "عليه", "عليها", "ذلكم", "هناك"}


def norm_ar(t):
    t = AR_DIACRITICS.sub("", t)
    t = re.sub(r"[آأإ]", "ا", t)
    return t.replace("ى", "ي").replace("ة", "ه")


def content_words(text):
    toks = {w.lower() for w in WORD_RE.findall(norm_ar(str(text)))}
    return {w for w in toks if w not in EN_STOP and w not in AR_STOP and len(w) > 1}


article_text = {}
for c in chunks.itertuples():
    article_text[(c.law, c.language, c.article_number)] = c.text


def echo(question, row):
    gold = set()
    for a in row["answer_article_numbers"]:
        gold |= content_words(article_text[(row["answer_law"], row["answer_language"], str(a))])
    q = content_words(question)
    return len(q & gold) / len(q) if q else 0.0


rows = []
for r in v3:
    rows.append({
        "question_id": r["question_id"],
        "lang": r["question_language"],
        "v2_echo": echo(r["question_text_statute"], r),
        "v3_echo": echo(r["question_text"], r),
    })
ec = pd.DataFrame(rows)
ec["delta"] = ec["v3_echo"] - ec["v2_echo"]

print("share of a question's content words that also appear in its own answer article")
print(f"  v2: mean {ec.v2_echo.mean():.3f}   median {ec.v2_echo.median():.3f}")
print(f"  v3: mean {ec.v3_echo.mean():.3f}   median {ec.v3_echo.median():.3f}")
print(f"  change: {ec.delta.mean():+.3f} mean\n")
print(ec.groupby("lang")[["v2_echo", "v3_echo", "delta"]].mean().round(3).to_string())
print(f"\nquestions where echo went UP (check these by hand): "
      f"{ec[ec.delta > 0].question_id.tolist() or 'none'}")
assert ec.v3_echo.mean() < ec.v2_echo.mean(), "rewrite did not reduce statute echo -- do not save"

share of a question's content words that also appear in its own answer article
  v2: mean 0.708   median 0.707
  v3: mean 0.285   median 0.273
  change: -0.424 mean

      v2_echo  v3_echo  delta
lang                         
ar      0.680    0.260 -0.420
en      0.737    0.309 -0.428

questions where echo went UP (check these by hand): ['q44', 'q55']


### The two questions the check flags

`q44` and `q55` come out with *higher* overlap in v3. Both were inspected by hand and both are fine — the metric misfires rather than the rewrite:

- **q55** — v2 used the statute's exact term **العطلة الأسبوعية**; v3 says **راحة في الأسبوع**. The statute's own word is gone, which is precisely the intended change. The ratio rose only because the question is nine words long, so a single shared word moves it a lot.
- **q44** — v3 replaced the definite statute phrase **الإجازة السنوية** with the indefinite **إجازة سنوية**. There is no lay synonym for "annual leave" that a real person would actually type; inventing one would make the question artificial and defeat the point of the rewrite.

Two known limitations of this metric, worth stating so it is not over-trusted: it does not strip Arabic clitics (`وهل` is a different token from `هل`, so some function words survive stopword removal), and on short questions the denominator is small enough that one word swings the ratio. It is a guide for spotting rewrites that failed, not a target to optimise against — driving it lower by avoiding words people genuinely use would make the eval set *less* realistic, not more.

## Save

`eval_v2.jsonl` is not touched.

In [5]:
V3_JSONL = PROJECT / "eval" / "eval_v3.jsonl"
V3_CSV = PROJECT / "eval" / "eval_v3.csv"

with open(V3_JSONL, "w", encoding="utf-8") as f:
    for r in v3:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

df = pd.DataFrame(v3)
df.to_csv(V3_CSV, index=False, encoding="utf-8-sig")

assert V2_PATH.read_text(encoding="utf-8") == "".join(
    json.dumps(r, ensure_ascii=False) + "\n" for r in v2), "eval_v2.jsonl was modified -- it must not be"

print(f"wrote {V3_JSONL.name} and {V3_CSV.name} ({len(v3)} questions)")
print(f"eval_v2.jsonl left intact: {V2_PATH.exists()}")
print("\nsample:")
for r in v3[:2] + v3[15:17] + v3[43:45]:
    print(f"  {r['question_id']} [{r['question_language']}] {r['answer_law']} art "
          f"{','.join(r['answer_article_numbers'])}")
    print(f"      v2: {r['question_text_statute']}")
    print(f"      v3: {r['question_text']}")

wrote eval_v3.jsonl and eval_v3.csv (60 questions)
eval_v2.jsonl left intact: True

sample:
  q01 [en] ssl art 25
      v2: What benefits does Work Injury Insurance provide to an injured worker?
      v3: I was injured while at work. What does social security actually cover for me?
  q02 [en] ssl art 26
      v2: Does the medical care provided under Work Injury Insurance include transport costs and prosthetic devices?
      v3: I need an artificial limb and I have to travel for treatment. Will those costs be paid for me?
  q16 [ar] ssl art 24
      v2: من يتحمل اشتراك تأمين إصابات العمل، العامل أم صاحب العمل؟
      v3: هل يُخصم من راتبي شيء مقابل التأمين ضد إصابات العمل، أم يدفعه صاحب العمل وحده؟
  q17 [ar] ssl art 28
      v2: إلى متى تبقى المؤسسة مسؤولة عن علاج المصاب بعد إصابة العمل؟
      v3: أُصبت في العمل وما زلت أتعالج. إلى متى تستمر المؤسسة في تغطية علاجي؟
  q44 [ar] labor96 art 61,62,63,64
      v2: ما أحكام الإجازة السنوية من حيث مدتها وجواز تجزئتها ومصيرها إذا لم تُستعمل؟
  

## Next

v3 is a measuring instrument, not a result. The immediate follow-on is to re-score the seven Phase 13 embedding models on it — near-free, because `eval/emb_cache/` already holds every model's corpus vectors and only the 60 new questions need embedding.

Two things that re-run will tell us:

1. **How much of Phase 13's 100% was the phrasing.** v2 and v3 have identical ground truth, so the drop is a direct measurement of the inflation, not a guess.
2. **Whether the model ranking survives.** If Cohere and Gemini stay level, Phase 13's decision stands on firmer evidence than it currently does. If the field spreads out, the eval set has its discriminating power back and the pending RRF `k`/`w` and BM25 `k1`/`b` re-tune finally has a ruler worth trusting.

Only after that is it worth asking whether a query parser earns its cost — with v3 in hand there is finally something for it to improve, and a way to prove it did.